# 基本面綜合評分篩選 — 找出 ≥90 分的股票

**評分邏輯**（共 16 項，每項通過 +1；第 1~12 項與 fundamental.html 一致，13~16 為「連續成長」項目，fundamental.html 尚未加入）

| # | 欄位 | 說明 |
|---|------|------|
| 1 | 財報健康 | |
| 2 | 財報強勢 | |
| 3 | 財報動能強 | |
| 4 | EPS_前8季全為正 | |
| 5 | EPS_維持 | |
| 6 | 毛利率_維持 | |
| 7 | EPS_創新高 | |
| 8 | 毛利率_創新高 | |
| 9 | EPS_成長 | |
| 10 | EPS_YoY成長 | |
| 11 | 毛利率成長 | |
| 12 | 毛利率YoY成長 | |
| 13 | EPS_YoY連4季 | 單季 EPS 比去年同季高，且連續 ≥ 4 季 |
| 14 | 營益_YoY連4季 | 營業利益比去年同季高，且連續 ≥ 4 季（本業） |
| 15 | TTM_EPS連4季增 | 近四季 EPS 加總連續 ≥ 4 季墊高 |
| 16 | 業外占比低 | 最新一季 業外收支 / 稅前淨利 < 30%（且稅前淨利 > 0） |

分數 = pass數 / 16 × 100，≥90 分表示至少通過 15 項。

連續成長用「差額 > 0」判斷（不用百分比，避免轉虧為盈的基期效應）；季別不連續（缺季）時視為中斷。

**資料來源**
- `Result/StockProfitability_all.xlsx`　→ 評分欄位取**最新一季**
- `Result/checkall/{date}.xlsx`　→ 補估值欄位（本益比、殖利率、淨值倍率、現價…）

**型態標籤**（另外輸出，不計入 score）

| 欄位 | 條件 |
|------|------|
| Type_存股 | 非景氣循環股、近 20 季 EPS 全正、近 12 季毛利率標準差 < 5、平均營益率 > 10%、業外占比 < 30%、EPS 4 年 CAGR > 0 |
| Type_轉機 | ① 虧轉盈：前 4 季至少虧 2 季、近 2 季都賺、TTM 轉正　② 毛利回升：一年前 TTM EPS 低於兩年前，毛利率自近 8 季低點回升 ≥ 5 個百分點、連 2 季 QoQ 上升、EPS YoY 成長 |
| Type_加速 | TTM EPS 年增率連 3 季走高，且最新 ≥ 20% |
| Type_毛利結構 | 近 2 季毛利率都高於之前 20 季的最高值 |
| 景氣循環股 | 歷史 ≥ 40 季、虧損季比例 ≥ 5%、毛利率 P90－P10 ≥ 15 個百分點 |
| Type_景氣位置 | 僅景氣循環股：最新毛利率在自身歷史的百分位　≤30 谷底／谷底回升，≥80 高峰，其餘中段 |
| Type_警訊 | 警訊數量（說明見「警訊說明」）：營收增但毛利率連 3 季 YoY 降、業外占比 > 50%、毛利率連 2 季低於 8 季均、動能轉弱、EPS 連 2 季 YoY 衰退 |

⚠️ score 高 + 景氣循環股「高峰」＝ 可能是景氣頂點，不是買點。

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── 設定 ──────────────────────────────────────────────────────────
DATE         = '2026-10-21'          # ← 改這裡換日期
SCORE_CUTOFF = 70                    # ← 改這裡換門檻（0~100）

PROFIT_FILE  = Path('Result/StockProfitability_all.xlsx')
CHECKALL_FILE = Path(f'Result/checkall/{DATE}.xlsx')

# ── 評分欄位 ──────────────────────────────────────────────────────
SCORE_COLS = [
    '財報健康', '財報強勢', '財報動能強',
    'EPS_前8季全為正',
    'EPS_維持', '毛利率_維持',
    'EPS_創新高', '毛利率_創新高',
    'EPS_成長', 'EPS_YoY成長',
    '毛利率成長', '毛利率YoY成長',
    'EPS_YoY連4季', '營益_YoY連4季', 'TTM_EPS連4季增', '業外占比低',
]
N = len(SCORE_COLS)  # 16

GROWTH_STREAK = 4      # ← 連續成長門檻（季）
NONOP_MAX     = 0.3    # ← 業外收支 / 稅前淨利 上限

print(f'日期：{DATE}　門檻：≥{SCORE_CUTOFF} 分')

日期：2026-10-21　門檻：≥70 分


In [2]:
# ── 1. 讀取財報歷史，取每支股票最新一季 ──────────────────────────
print('讀取 StockProfitability_all.xlsx ...')
df_profit = pd.read_excel(PROFIT_FILE, dtype={'stock_number': str})
df_profit['stock_number'] = df_profit['stock_number'].str.zfill(4)

# 最新一季（season_key 最大）
df_latest = (
    df_profit
    .sort_values('season_key')
    .groupby('stock_number', as_index=False)
    .last()
)

print(f'共 {len(df_latest)} 支股票，最新季範圍：{df_latest["季別"].min()} ~ {df_latest["季別"].max()}')

讀取 StockProfitability_all.xlsx ...
共 1972 支股票，最新季範圍：114.3Q ~ 115.2Q


In [3]:
# ── 1-1. 連續成長（YoY / TTM / 業外）─────────────────────────────
# 用完整歷史計算，依 season_key 排序；季別不連續時視為中斷
def _streak(s):
    """回傳到每一列為止，連續 > 0 的季數（NaN 視為中斷）。"""
    pos = (s > 0).astype(int)
    return pos.groupby((pos == 0).cumsum()).cumsum()

df_g = df_profit.copy()
for col in ['EPS(元)', '營業利益', '業外收支', '稅前淨利']:
    df_g[col] = pd.to_numeric(df_g[col], errors='coerce')
df_g['_qidx'] = df_g['year'].astype(int) * 4 + df_g['q'].astype(int)
df_g = df_g.sort_values(['stock_number', '_qidx']).reset_index(drop=True)
g = df_g.groupby('stock_number')

# 去年同季差額（只有剛好相隔 4 季才算，缺季 -> NaN）
yoy_ok = (g['_qidx'].diff(4) == 4)
df_g['_eps_yoy'] = g['EPS(元)'].diff(4).where(yoy_ok)
df_g['_oi_yoy']  = g['營業利益'].diff(4).where(yoy_ok)

# TTM EPS（近 4 季加總，需連續 4 季）與其季增
ttm_ok = (g['_qidx'].diff(3) == 3)
df_g['TTM_EPS'] = g['EPS(元)'].transform(lambda s: s.rolling(4).sum()).where(ttm_ok)
df_g['_ttm_d'] = df_g.groupby('stock_number')['TTM_EPS'].diff().where(g['_qidx'].diff() == 1)

df_g['EPS_YoY連續季數'] = df_g.groupby('stock_number')['_eps_yoy'].transform(_streak)
df_g['營益_YoY連續季數'] = df_g.groupby('stock_number')['_oi_yoy'].transform(_streak)
df_g['TTM_EPS連續季數'] = df_g.groupby('stock_number')['_ttm_d'].transform(_streak)

# 業外占比：稅前淨利 <= 0 時不計算（不通過）
pretax = df_g['稅前淨利']
df_g['業外占比'] = (df_g['業外收支'].fillna(0) / pretax).where(pretax > 0).round(3)

GROWTH_COLS = ['TTM_EPS', 'EPS_YoY連續季數', '營益_YoY連續季數', 'TTM_EPS連續季數', '業外占比']
df_g_latest = df_g.groupby('stock_number', as_index=False).last()[['stock_number'] + GROWTH_COLS]

df_latest = df_latest.drop(columns=[c for c in GROWTH_COLS if c in df_latest.columns])                      .merge(df_g_latest, on='stock_number', how='left')

df_latest['EPS_YoY連4季']   = (df_latest['EPS_YoY連續季數'] >= GROWTH_STREAK).astype(int)
df_latest['營益_YoY連4季']  = (df_latest['營益_YoY連續季數'] >= GROWTH_STREAK).astype(int)
df_latest['TTM_EPS連4季增'] = (df_latest['TTM_EPS連續季數'] >= GROWTH_STREAK).astype(int)
df_latest['業外占比低']     = (df_latest['業外占比'] < NONOP_MAX).astype(int)

for col in ['EPS_YoY連4季', '營益_YoY連4季', 'TTM_EPS連4季增', '業外占比低']:
    print(f'  {col:<14} 通過 {int(df_latest[col].sum()):>4} 支')


  EPS_YoY連4季     通過  455 支
  營益_YoY連4季      通過  362 支
  TTM_EPS連4季增    通過  458 支
  業外占比低          通過 1299 支


In [4]:
# ── 2. 計算評分 ───────────────────────────────────────────────────
for col in SCORE_COLS:
    if col not in df_latest.columns:
        print(f'  ⚠️  欄位缺失：{col}，補 0')
        df_latest[col] = 0
    df_latest[col] = pd.to_numeric(df_latest[col], errors='coerce').fillna(0).astype(int)

df_latest['pass_count'] = df_latest[SCORE_COLS].sum(axis=1)
df_latest['score']      = (df_latest['pass_count'] / N * 100).round(1)

print('評分分布：')
print(df_latest['score'].describe().to_string())

評分分布：
count    1972.000000
mean       45.322059
std        24.768543
min         0.000000
25%        25.000000
50%        43.800000
75%        62.500000
max       100.000000


In [ ]:
# ── 2-1. 型態標籤（與 score 分開，不影響 16 項評分）────────────────
# 存股 / 轉機 / 加速 / 毛利結構 / 景氣循環股 + 景氣位置 / 警訊
# 用完整歷史（df_g）計算，季別不連續時該型態不成立
STABLE_Q      = 20     # ← 存股：近 N 季 EPS 全為正
STABLE_GM_STD = 5      # ← 存股：近 12 季毛利率標準差上限（百分點）
STABLE_OPM    = 10     # ← 存股：近 12 季平均營益率下限（%）
TURN_GM_UP    = 5      # ← 轉機：毛利率自近 8 季低點回升（百分點）
ACCEL_MIN     = 0.2    # ← 加速：最新 TTM EPS 年增率下限
CYC_LOSS      = 0.05   # ← 景氣循環股：歷史虧損季比例下限
CYC_GM_RANGE  = 15     # ← 景氣循環股：歷史毛利率 P90-P10 下限（百分點）
CYC_MIN_Q     = 40     # ← 景氣循環股：至少要有 N 季歷史才判斷

df_t = df_g.copy()
df_t['毛利率'] = pd.to_numeric(df_t['毛利率'], errors='coerce')
df_t['營業收入'] = pd.to_numeric(df_t['營業收入'], errors='coerce')
df_t['營益率'] = pd.to_numeric(df_t['營益率'].astype(str).str.rstrip('%'), errors='coerce')

def _contig(s, n):
    """最後 n 季的季別是否連續"""
    return len(s) >= n and s.iloc[-1] - s.iloc[-n] == n - 1

def _types(t):
    t = t.reset_index(drop=True)
    qi, eps, gm = t['_qidx'], t['EPS(元)'], t['毛利率']
    r = {}

    # 共用指標
    l12 = t.tail(12)
    r['GM_std_12'] = l12['毛利率'].std() if _contig(qi, 12) else np.nan
    r['OPM_avg_12'] = l12['營益率'].mean() if _contig(qi, 12) else np.nan
    ttm = t['TTM_EPS']
    r['TTM_EPS_YoY%'] = (ttm.iloc[-1] / ttm.iloc[-5] - 1) * 100 \
        if _contig(qi, 5) and ttm.iloc[-5] > 0 else np.nan
    rev = t['營業收入']
    r['營收_TTM_YoY%'] = (rev.tail(4).sum() / rev.iloc[-8:-4].sum() - 1) * 100 \
        if _contig(qi, 8) and rev.iloc[-8:-4].sum() > 0 else np.nan
    r['EPS_5年CAGR%'] = ((ttm.iloc[-1] / ttm.iloc[-17]) ** (1 / 4) - 1) * 100 \
        if _contig(qi, 17) and ttm.iloc[-17] > 0 and ttm.iloc[-1] > 0 else np.nan

    # 存股：近 20 季全賺、毛利率穩、本業賺、近 4 年 EPS 有成長
    pre12 = l12['稅前淨利'].sum()
    nonop12 = l12['業外收支'].fillna(0).sum() / pre12 if pre12 > 0 else np.nan
    r['Type_存股'] = int(
        _contig(qi, STABLE_Q) and (eps.tail(STABLE_Q) > 0).all()
        and r['GM_std_12'] < STABLE_GM_STD and r['OPM_avg_12'] > STABLE_OPM
        and abs(nonop12) < NONOP_MAX and r['EPS_5年CAGR%'] > 0)

    # 轉機：①虧轉盈（前 4 季至少虧 2 季、近 2 季都賺、TTM 轉正）②先衰退（一年前 TTM EPS < 兩年前）、毛利率自近 8 季低點回升且連 2 季 QoQ 上升、EPS YoY 成長
    why = []
    if _contig(qi, 6) and (eps.iloc[-6:-2] <= 0).sum() >= 2 and (eps.tail(2) > 0).all() and ttm.iloc[-1] > 0:
        why.append('虧轉盈')
    if _contig(qi, 9) and ttm.iloc[-5] < ttm.iloc[-9] and gm.iloc[-1] - gm.tail(8).min() >= TURN_GM_UP \
            and (gm.diff().tail(2) > 0).all() and t['_eps_yoy'].iloc[-1] > 0:
        why.append('毛利回升')
    r['Type_轉機'] = int(bool(why))
    r['轉機說明'] = ' / '.join(why)

    # 加速：TTM EPS 年增率連 3 季走高（用 TTM 避開季節性）
    acc = 0
    if _contig(qi, 7) and (ttm.iloc[-7:-4] > 0).all():
        g3 = ttm.iloc[-3:].values / ttm.iloc[-7:-4].values - 1
        acc = int(g3[0] < g3[1] < g3[2] and g3[2] >= ACCEL_MIN)
    r['Type_加速'] = acc

    # 毛利結構：近 2 季毛利率都高於之前 20 季的最高值
    r['Type_毛利結構'] = int(_contig(qi, 22) and gm.tail(2).min() > gm.iloc[-22:-2].max())

    # 景氣循環股 + 景氣位置（最新毛利率在自身歷史的百分位）
    gm_all = gm.dropna()
    cyc = len(gm_all) >= CYC_MIN_Q and (eps <= 0).mean() >= CYC_LOSS \
        and gm_all.quantile(.9) - gm_all.quantile(.1) >= CYC_GM_RANGE
    r['景氣循環股'] = int(cyc)
    r['毛利率_歷史百分位'] = round((gm_all < gm.iloc[-1]).mean() * 100) if len(gm_all) >= 20 and pd.notna(gm.iloc[-1]) else np.nan
    pos = '-'
    if cyc and pd.notna(r['毛利率_歷史百分位']):
        p, up = r['毛利率_歷史百分位'], gm.diff().iloc[-1] > 0
        pos = ('谷底回升' if up else '谷底') if p <= 30 else ('高峰' if p >= 80 else '中段')
    r['Type_景氣位置'] = pos

    # 警訊（持股檢查用）
    w = []
    if r['營收_TTM_YoY%'] > 0 and _contig(qi, 7) and (t['GM_YoY'].tail(3) < 0).all():
        w.append('營收增但毛利率連3季YoY降')
    if t['業外占比'].iloc[-1] > 0.5:
        w.append('業外占比>50%')
    if len(t) >= 2 and (gm.tail(2) < t['GM_MA_8'].tail(2)).all():
        w.append('毛利率連2季低於8季均')
    if len(t) >= 2 and t['財報動能強'].iloc[-2] == 1 and t['財報動能強'].iloc[-1] == 0:
        w.append('動能轉弱')
    if (t['_eps_yoy'].tail(2) < 0).all() and t['_eps_yoy'].tail(2).notna().all():
        w.append('EPS連2季YoY衰退')
    r['Type_警訊'] = len(w)
    r['警訊說明'] = ' / '.join(w)
    return pd.Series(r)

df_types = df_t.groupby('stock_number').apply(_types).reset_index()
# 存股排除景氣循環股（近 5 年穩不代表整個循環都穩）
df_types['Type_存股'] = df_types['Type_存股'] * (1 - df_types['景氣循環股'])
TYPE_COLS = [c for c in df_types.columns if c != 'stock_number']

df_latest = df_latest.drop(columns=[c for c in TYPE_COLS if c in df_latest.columns]) \
                     .merge(df_types, on='stock_number', how='left')

for col in ['Type_存股', 'Type_轉機', 'Type_加速', 'Type_毛利結構', '景氣循環股']:
    print(f'  {col:<10} {int(df_latest[col].sum()):>4} 支')
print('  景氣位置：', df_latest['Type_景氣位置'].value_counts().to_dict())
print('  警訊數  ：', df_latest['Type_警訊'].value_counts().sort_index().to_dict())

In [5]:
# ── 3. 讀取 checkall，補估值欄位 ─────────────────────────────────
print(f'讀取 {CHECKALL_FILE} ...')
df_check = pd.read_excel(CHECKALL_FILE, dtype={'stock_number': str})
df_check['stock_number'] = df_check['stock_number'].str.zfill(4)

# 要從 checkall 補的欄位
VAL_COLS = [
    'stock_number',
    'Type0',      # 股票名稱
    'Type1',      # 產業別
    'Type2',      # 上市/上櫃
    'now_price',  # 現價
    '_quote',     # 漲跌幅%
    '本益比', '同業平均本益比',
    '淨值倍率', '殖利率',
    '每股營收(元)',
    '營業收入-上月比較增減(%)',
    '營業收入-去年同月增減(%)',
    '累計營業收入-前期比較增減(%)',
    '總市值',
]
VAL_COLS_EXIST = [c for c in VAL_COLS if c in df_check.columns]
df_val = df_check[VAL_COLS_EXIST].copy()

print(f'checkall 共 {len(df_val)} 支股票')

讀取 Result\checkall\2026-10-21.xlsx ...
checkall 共 1944 支股票


In [6]:
# ── 4. Join ───────────────────────────────────────────────────────
df_score = df_latest[['stock_number','季別','EPS(元)','毛利率','EPS_MA_8','GM_MA_8',
                       'pass_count','score'] + GROWTH_COLS + SCORE_COLS + TYPE_COLS].copy()

df_merged = df_score.merge(df_val, on='stock_number', how='left')

# 篩選 ≥ cutoff
df_high = df_merged[df_merged['score'] >= SCORE_CUTOFF].copy()
df_high = df_high.sort_values(['score', 'TTM_EPS連續季數'], ascending=False).reset_index(drop=True)

print(f'\n≥{SCORE_CUTOFF} 分的股票：{len(df_high)} 支')


≥70 分的股票：324 支


In [7]:
# ── 5. 展示結果 ───────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# 轉換估值欄位為數值
for col in ['本益比','同業平均本益比','淨值倍率','殖利率','now_price','EPS(元)','毛利率']:
    if col in df_high.columns:
        df_high[col] = pd.to_numeric(df_high[col], errors='coerce')

# 估值判斷
def pe_judge(row):
    pe, ipe = row.get('本益比'), row.get('同業平均本益比')
    if pd.isna(pe) or pd.isna(ipe) or ipe == 0: return '-'
    ratio = pe / ipe
    if ratio < 0.85: return '🟢 偏低'
    if ratio > 1.15: return '🔴 偏高'
    return '🟡 合理'

df_high['本益比判斷'] = df_high.apply(pe_judge, axis=1)

# 顯示欄位
SHOW = ['stock_number','Type0','Type1','score','pass_count',
        '季別','EPS(元)','毛利率','TTM_EPS',
        'EPS_YoY連續季數','營益_YoY連續季數','TTM_EPS連續季數','業外占比',
        'now_price','本益比','同業平均本益比','本益比判斷','淨值倍率','殖利率',
        '累計營業收入-前期比較增減(%)',
        'Type_存股','Type_轉機','Type_加速','Type_毛利結構','景氣循環股','Type_景氣位置','Type_警訊','警訊說明']
SHOW = [c for c in SHOW if c in df_high.columns]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 200)

df_high[SHOW]

,stock_number,Type0,Type1,score,pass_count,季別,EPS(元),毛利率,TTM_EPS,EPS_YoY連續季數,營益_YoY連續季數,TTM_EPS連續季數,業外占比,now_price,本益比,同業平均本益比,本益比判斷,淨值倍率,殖利率,累計營業收入-前期比較增減(%)
0,3017,奇鋐,電腦及週邊設備業,100.00,16,115.2Q,24.37,32.57,75.22,30,30,30,0.03,3235.00,43.12,25.52,🔴 偏高,23.01,0.65,99.34
1,6446,藥華藥,生技醫療業,100.00,16,115.2Q,6.13,94.23,19.90,13,11,13,0.01,1175.00,57.54,33.18,🔴 偏高,11.92,0.22,60.27
2,8462,柏文,運動休閒,100.00,16,115.2Q,3.25,35.30,10.63,13,13,13,-0.03,163.50,15.66,20.45,🟢 偏低,5.46,4.45,20.32
3,2383,台光電,電子零組件業,100.00,16,115.2Q,27.55,33.85,64.27,12,13,12,0.00,4990.00,78.03,51.30,🔴 偏高,30.65,0.50,62.74
4,3044,健鼎,電子零組件業,100.00,16,115.2Q,7.41,30.09,23.34,12,12,12,0.02,583.00,24.98,51.30,🟢 偏低,5.50,2.18,24.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,6547,高端疫苗,生技醫療業,75.00,12,115.2Q,0.07,81.55,-0.51,1,2,1,0.04,50.60,0.00,47.03,🟢 偏低,4.68,0.00,-8.13
320,8050,廣積,電腦及週邊設備業,75.00,12,115.2Q,1.62,25.87,3.67,1,1,1,0.28,63.30,24.01,25.58,🟡 合理,1.71,3.39,-14.14
321,8284,三竹,資訊服務業,75.00,12,115.2Q,1.86,27.61,6.68,1,1,1,0.01,68.00,10.38,17.18,🟢 偏低,2.41,5.89,10.64
322,8421,旭源,其他,75.00,12,115.2Q,0.26,21.84,0.44,1,2,1,-2.11,14.95,33.57,41.34,🟢 偏低,1.10,0.00,9.07


In [8]:
# ── 6. 每項通過細節 ───────────────────────────────────────────────
print(f'\n各評分項目通過率（≥{SCORE_CUTOFF} 分的 {len(df_high)} 支）：')
for col in SCORE_COLS:
    if col in df_high.columns:
        n = df_high[col].sum()
        print(f'  {col:<18} {int(n):>4} / {len(df_high)}  ({n/len(df_high)*100:.0f}%)')


各評分項目通過率（≥70 分的 324 支）：
  財報健康                323 / 324  (100%)
  財報強勢                229 / 324  (71%)
  財報動能強               320 / 324  (99%)
  EPS_前8季全為正          194 / 324  (60%)
  EPS_維持              324 / 324  (100%)
  毛利率_維持              323 / 324  (100%)
  EPS_創新高             278 / 324  (86%)
  毛利率_創新高             255 / 324  (79%)
  EPS_成長              312 / 324  (96%)
  EPS_YoY成長           324 / 324  (100%)
  毛利率成長               294 / 324  (91%)
  毛利率YoY成長            320 / 324  (99%)
  EPS_YoY連4季          200 / 324  (62%)
  營益_YoY連4季           175 / 324  (54%)
  TTM_EPS連4季增         200 / 324  (62%)
  業外占比低               293 / 324  (90%)


In [9]:
# ── 7. 產業分布 ───────────────────────────────────────────────────
if 'Type1' in df_high.columns:
    print(f'\n產業分布（≥{SCORE_CUTOFF} 分）：')
    print(df_high['Type1'].value_counts().to_string())


產業分布（≥70 分）：
電子零組件業      57
半導體業        56
其他電子業       18
鋼鐵工業        18
電腦及週邊設備業    16
通信網路業       15
其他          15
電機機械        14
生技醫療業       13
化學工業        13
綠能環保        10
光電業          8
電子通路業        8
塑膠工業         7
建材營造         6
資訊服務業        6
紡織纖維         5
航運業          4
金融業          3
運動休閒         3
數位雲端         3
食品工業         3
汽車工業         3
油電燃氣業        3
觀光事業         3
農業科技業        3
橡膠工業         2
玻璃陶瓷         1
金融保險         1
文化創意業        1
居家生活         1


In [ ]:
# ── 7-1. 型態清單（全部股票，不受 SCORE_CUTOFF 限制）────────────────
df_all = df_merged.copy()
for col in ['本益比', '殖利率', 'now_price']:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

TYPE_SHOW = ['stock_number', 'Type0', 'Type1', 'score', 'TTM_EPS', 'TTM_EPS_YoY%', '營收_TTM_YoY%',
             'EPS_5年CAGR%', 'GM_std_12', 'OPM_avg_12', '毛利率_歷史百分位', 'Type_景氣位置',
             '轉機說明', '警訊說明', 'now_price', '本益比', '殖利率']
TYPE_SHOW = [c for c in TYPE_SHOW if c in df_all.columns]

def show_type(title, mask, sort='score', n=30):
    x = df_all[mask].sort_values(sort, ascending=False)
    print(f'\n■ {title}：{len(x)} 支（顯示前 {min(n, len(x))}）')
    display(x[TYPE_SHOW].head(n))

show_type('存股（近20季全賺、毛利穩、本業賺、4年EPS成長）', df_all['Type_存股'] == 1, 'EPS_5年CAGR%')
show_type('轉機（虧轉盈 / 衰退後毛利回升）', df_all['Type_轉機'] == 1)
show_type('加速（TTM EPS 年增率連3季走高）', df_all['Type_加速'] == 1)
show_type('毛利結構（近2季毛利率突破前20季高點）', df_all['Type_毛利結構'] == 1)
show_type('景氣循環股・谷底回升', df_all['Type_景氣位置'] == '谷底回升', '毛利率_歷史百分位')
show_type('⚠️ 高分但景氣循環股在高峰（score ≥ SCORE_CUTOFF）',
          (df_all['Type_景氣位置'] == '高峰') & (df_all['score'] >= SCORE_CUTOFF))
show_type('⚠️ 高分但有警訊（score ≥ SCORE_CUTOFF）',
          (df_all['Type_警訊'] > 0) & (df_all['score'] >= SCORE_CUTOFF))

In [10]:
# ── 8. 輸出 Excel ─────────────────────────────────────────────────
out_path = Path(f'Result/fundamental_score_{DATE}_top{SCORE_CUTOFF}.xlsx')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path1 = Path(f'Result/fundamental_score_all.xlsx')
out_path1.parent.mkdir(parents=True, exist_ok=True)

# 輸出完整欄位
df_high.to_excel(out_path, index=False)
print(f'已輸出：{out_path}　（{len(df_high)} 支股票）')

df_merged.to_excel(out_path1, index=False)
print(f'已輸出：{out_path1}　（{len(df_merged)} 支股票）')


已輸出：Result\fundamental_score_2026-10-21_top70.xlsx　（324 支股票）
已輸出：Result\fundamental_score_all.xlsx　（1972 支股票）


In [11]:
# ── 9. 不同門檻的分布（參考用）──────────────────────────────────
print('各分數門檻對應股票數：')
for cutoff in [100, 90, 80, 70, 60, 50]:
    n = (df_merged['score'] >= cutoff).sum()
    bar = '█' * (n // 10)
    print(f'  ≥{cutoff:3d} 分：{n:4d} 支  {bar}')

各分數門檻對應股票數：
  ≥100 分：  48 支  ████
  ≥ 90 分：  81 支  ████████
  ≥ 80 分： 224 支  ██████████████████████
  ≥ 70 分： 324 支  ████████████████████████████████
  ≥ 60 分： 586 支  ██████████████████████████████████████████████████████████
  ≥ 50 分： 912 支  ███████████████████████████████████████████████████████████████████████████████████████████


In [12]:
print(df_high[SHOW]['stock_number'].tolist())


['3017', '6446', '8462', '2383', '3044', '3167', '2330', '5274', '6782', '8021', '2010', '3081', '3357', '6183', '1560', '2027', '5607', '2301', '3264', '1714', '6625', '7713', '8271', '1590', '2059', '2451', '3265', '3711', '4721', '4771', '5289', '5340', '6173', '6257', '6274', '6291', '6417', '6672', '6727', '6753', '6903', '6967', '7810', '8028', '8083', '8109', '8299', '8415', '3583', '4506', '3551', '6023', '3219', '4764', '4950', '5475', '5521', '1513', '4304', '1305', '1802', '2006', '2344', '2408', '2441', '2481', '3229', '3305', '3550', '4541', '4923', '4971', '5498', '6133', '6265', '6426', '6435', '6716', '6776', '8046', '8131', '3029', '6138', '2345', '2404', '3049', '3048', '4540', '6894', '1459', '5511', '6931', '8432', '1303', '1304', '1416', '1514', '1815', '2031', '2413', '3037', '3141', '3624', '4503', '4555', '4707', '4934', '5291', '6139', '6197', '6442', '6870', '8039', '8150', '3026', '6005', '7711', '2478', '6151', '6525', '7842', '3653', '6606', '6223', '6805',

In [13]:
print(df_high[df_high['EPS_前8季全為正']==1]['stock_number'].tolist())

['3017', '6446', '8462', '2383', '3044', '3167', '2330', '5274', '6782', '8021', '2010', '3081', '3357', '6183', '1560', '2027', '5607', '2301', '3264', '1714', '6625', '7713', '8271', '1590', '2059', '2451', '3265', '3711', '4721', '4771', '5289', '5340', '6173', '6257', '6274', '6291', '6417', '6672', '6727', '6753', '6903', '6967', '7810', '8028', '8083', '8109', '8299', '8415', '3583', '4506', '3551', '6023', '1513', '2006', '2441', '2481', '3305', '6435', '6716', '6776', '3029', '6138', '2345', '2404', '6894', '5511', '8432', '1416', '1514', '3037', '4707', '6139', '6197', '6442', '6870', '8039', '3026', '6005', '7711', '2478', '6151', '6525', '7842', '3653', '6606', '6223', '6805', '2368', '3030', '6669', '6739', '6944', '5234', '5243', '5439', '2308', '8210', '2467', '3260', '3293', '3702', '5209', '1232', '1256', '1597', '1720', '2207', '2376', '2455', '2471', '2476', '5434', '6530', '6584', '6831', '8358', '8436', '1709', '2049', '2069', '2420', '2762', '3045', '3645', '5538',